# 5. MCH mCG scale

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed
from statsmodels.tsa.stattools import acf

import pysam
import cooler
import anndata
import scanpy as sc
from sklearn.cluster import KMeans

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/mCH_mCG_scale/'

In [ ]:
ct = 'c5'
allc_path = f'{indir}merged_allc/L1/CHN/{ct}.allc.tsv.gz'


In [ ]:
reslist = pd.Index([1, 5, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000])


In [ ]:
def num2str(num):
    if num>=1e6:
        num_str = f'{int(num/1e6)}M'
    elif num>=1e3:
        num_str = f'{int(num/1e3)}k'
    else:
        num_str = f'{num}'
    return num_str


In [ ]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
rm_list = []
for bed_path in [f'{REF_ROOT}/hg38/fasta/hg38.altseq.bed', 
                 f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz']:
    bed = pd.read_csv(bed_path, sep='\t', header=None, index_col=None)
    bed = {chrom:bed.loc[bed[0]==chrom, [1,2]].values for chrom in chrom_sizes.index}
    rm_list.append(bed)
    

In [ ]:
chrom, start, end = 'chr1', 5000000, 35000000

data_all = {xx:[] for xx in 'ACGT'}
result = {xx:[] for xx in 'ACGT'}
with pysam.TabixFile(allc_path) as allc:
    allc_lines = allc.fetch(chrom, start, end)
    for line in allc_lines:
        _, pos, _, context, mc, cov, *_ = line.split("\t")
        if context[1]!='N':
            data_all[context[1]].append([pos, mc, cov])

for context in 'ACGT':
    print(context)
    data = data_all[context]
    data = pd.DataFrame(data, columns=['pos', 'mc', 'cov']).astype(int)

    posfilter = np.ones(data.shape[0]).astype(bool)
    for bed in rm_list:
        bedtmp = bed[chrom][np.logical_and(bed[chrom][:,0]<end, bed[chrom][:,1]>start)]
        for xx,yy in bedtmp:
            posfilter[np.logical_and(data['pos']>=xx, data['pos']<=yy)] = False

    data = data.loc[posfilter]
    data_mc = csr_matrix((data['mc'], (np.zeros(data.shape[0]), data['pos']-start-1)), shape=[1, end-start])
    data_cov = csr_matrix((data['cov'], (np.zeros(data.shape[0]), data['pos']-start-1)), shape=[1, end-start])

    for posres in reslist:
        npos = (end - start) // posres * posres
        tmp = data_mc.reshape((-1, posres)).sum(axis=1).A1 / data_cov.reshape((-1, posres)).sum(axis=1).A1
        # tmp[np.isnan(tmp)] = 0
        result[context].append(tmp)
        print(posres)


In [ ]:
fig, axes = plt.subplots(len(reslist), 1, figsize=(10, 5))
for i,xx in enumerate(result['A']):
    res = reslist[i]
    ax = axes[i]
    ax.plot(np.arange(len(xx)), xx, linewidth=np.sqrt(res/50000))
    vmin, vmax = np.percentile(xx[~np.isnan(xx)], 0), np.percentile(xx[~np.isnan(xx)], 99)
    ax.set_ylim([vmin, vmax])
    ax.set_ylabel(num2str(res))
    

In [ ]:
fig, axes = plt.subplots(len(reslist), 1, figsize=(10, 5))
for i,xx in enumerate(result['C']):
    res = reslist[i]
    ax = axes[i]
    ax.plot(np.arange(len(xx)), xx, linewidth=np.sqrt(res/50000))
    vmin, vmax = np.percentile(xx[~np.isnan(xx)], 0), np.percentile(xx[~np.isnan(xx)], 99)
    ax.set_ylim([vmin, vmax])
    ax.set_ylabel(num2str(res))
    

In [ ]:
fig, axes = plt.subplots(len(reslist), 1, figsize=(10, 5))
for i,xx in enumerate(result['T']):
    res = reslist[i]
    ax = axes[i]
    ax.plot(np.arange(len(xx)), xx, linewidth=np.sqrt(res/50000))
    vmin, vmax = np.percentile(xx[~np.isnan(xx)], 0), np.percentile(xx[~np.isnan(xx)], 99)
    ax.set_ylim([vmin, vmax])
    ax.set_ylabel(num2str(res))
    

In [ ]:
fig, axes = plt.subplots(len(reslist), 1, figsize=(10, 5))
for i,xx in enumerate(result['G']):
    res = reslist[i]
    ax = axes[i]
    ax.plot(np.arange(len(xx)), xx, linewidth=np.sqrt(res/50000))
    vmin, vmax = np.percentile(xx[~np.isnan(xx)], 0), np.percentile(xx[~np.isnan(xx)], 99)
    ax.set_ylim([vmin, vmax])
    ax.set_ylabel(num2str(res))
    

In [ ]:
fig, axes = plt.subplots(len(reslist)*4, 1, figsize=(10, 20))
for i,res in enumerate(reslist):
    for j,context in enumerate('CTAG'):
        ax = axes[i*4+j]
        xx = result[context][i]
        ax.plot(np.arange(len(xx)), xx, linewidth=np.sqrt(res/50000), color=f'C{j}')
        vmin, vmax = np.percentile(xx[~np.isnan(xx)], 0), np.percentile(xx[~np.isnan(xx)], 99)
        ax.set_ylim([vmin, vmax])
        ax.set_ylabel(f'C{context}\n{num2str(res)}')
    

In [ ]:
binres = 100000
bed = cooler.util.binnify(chrom_sizes, binres)
bed = bed.loc[bed['chrom'].isin(chrom_sizes.index)]
chunk_size = bed.index.max() // 48
bed['chunk'] = bed.index // chunk_size
print(bed.shape[0], bed['chunk'].max())


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import pysam
import cooler
import anndata
import scanpy as sc
from concurrent.futures import ProcessPoolExecutor, as_completed

def matrixch(allc_path, chunk):
    with pysam.TabixFile(allc_path) as allc:
        result = {context:{} for context in 'GATC'}
        for idx,(chrom,start,end) in chunk[['chrom', 'start', 'end']].iterrows():
            # npos = (end - start) // posres * posres
            # if npos==0:
            #     continue
            # region = f'{chrom}-{start}-{end}'
            data = {context:[] for context in 'GATCN'}
            allc_lines = allc.fetch(chrom, start, end)
            for line in allc_lines:
                _, pos, _, context, mc, cov, *_ = line.split("\t")
                data[context[1]].append([pos, mc, cov])
            del data['N']
            for context in 'GATC':
                tmp = pd.DataFrame(data[context], columns=['pos', 'mc', 'cov']).astype(int)            
                posfilter = np.ones(tmp.shape[0]).astype(bool)
                for bed in rm_list:
                    bedtmp = bed[chrom][np.logical_and(bed[chrom][:,0]<end, bed[chrom][:,1]>start)]
                    for xx,yy in bedtmp:
                        posfilter[np.logical_and(tmp['pos']>=xx, tmp['pos']<=yy)] = False
                tmp = tmp.loc[posfilter]
                tmp['pos'] = tmp['pos'] - start - 1
                # npos = (end - start)
                # data_mc = csr_matrix((tmp['mc'], (np.zeros(tmp.shape[0]), tmp['pos']-start-1)), shape=[1, npos])
                # data_cov = csr_matrix((tmp['cov'], (np.zeros(tmp.shape[0]), tmp['pos']-start-1)), shape=[1, npos])
                result[context][idx] = []
                for res in reslist[2:]:
                    result[context][idx].append(tmp.loc[tmp['pos']<res, 'mc'].sum() / tmp.loc[tmp['pos']<res, 'cov'].sum())
        for context in 'GATC':
            result[context] = pd.DataFrame(result[context])
        return result
    

In [ ]:
# chunk = bed.iloc[100:105]
# tmp = matrixch(allc_path, chunk)


In [ ]:
cpu = 24
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,chunk in bed.groupby('chunk'):
        future = executor.submit(
            matrixch,
            allc_path=allc_path, 
            chunk=chunk,
        )
        futures[future] = i

    result = {context:[] for context in 'GATC'}
    for future in as_completed(futures):
        idx = futures[future]
        tmp = future.result()
        for context in 'GATC':
            result[context].append(tmp[context])
        print(f'chunk{idx} finished')
        

In [ ]:
for context in 'GATC':
    result[context] = pd.concat(result[context], axis=1).sort_index(axis=1)
    

In [ ]:
for context in 'GATC':
    result[context].to_hdf(f'{outdir}mC_100kb.hdf', key=context)
    

In [ ]:
nlag = 100
palette = {context:c for context,c in zip('GATC', sns.color_palette('hls', len('GATC')))}
fig, axes = plt.subplots(3, 3, figsize=(6,6), dpi=300, sharex='all', sharey='all')
for j,res in enumerate(reslist[2:]):
    ax = axes.flatten()[j]
    for i,context in enumerate('GATC'):
        tmp = result[context].loc[j]
        tmp[tmp.isna()] = tmp.mean()
        tmp = np.mean([acf(tmp[bed['chrom']==c], nlags=nlag) for c in chrom_sizes.index[:22]], axis=0)
        ax.plot(np.arange(nlag+1), tmp, color=palette[context], label=f'C{context}')
    ax.set_title(num2str(res))
    ax.set_xticks(np.arange(0, nlag+1, 50))
    ax.set_xticklabels([num2str(xx*100000) for xx in np.arange(0, nlag+1, 50)])
    if j==0:
        ax.legend()

for ax in axes[-1]:
    ax.set_xlabel('Genome Distance')

for ax in axes[:, 0]:
    ax.set_ylabel('Pearson r')

fig.tight_layout()
fig.savefig(f'{outdir}plot/{ct}_autocorr_context_res.pdf', transparent=True)


In [ ]:
nlag = 50
palette = {res:c for res,c in zip(reslist, sns.color_palette('rainbow_r', len(reslist)))}
fig, axes = plt.subplots(2, 2, figsize=(6,6), dpi=300, sharex='all', sharey='all')
for i,context in enumerate('GATC'):
    ax = axes.flatten()[i]
    for j,res in enumerate(reslist[2:]):
        tmp = result[context].loc[j]
        tmp[tmp.isna()] = tmp.mean()
        tmp = np.mean([acf(tmp[bed['chrom']==c], nlags=nlag) for c in chrom_sizes.index[:22]], axis=0)
        ax.plot(np.arange(nlag+1), tmp, color=palette[res], label=num2str(res))
    ax.set_title(f'C{context}')
    ax.set_xticks(np.arange(0, nlag+1, 10))
    ax.set_xticklabels([num2str(xx*100000) for xx in np.arange(0, nlag+1, 10)])
    if i==0:
        ax.legend()

for ax in axes[-1]:
    ax.set_xlabel('Genome Distance')

for ax in axes[:, 0]:
    ax.set_ylabel('Pearson r')

fig.tight_layout()
fig.savefig(f'{outdir}plot/{ct}_autocorr_res_context.pdf', transparent=True)


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)

In [ ]:
i = 6
res = num2str(reslist[2:][i])
nlag = 50
result_all = []
for context in 'GATC':
    result = []
    for ct in L1_meta.index:
        tmp = pd.read_hdf(f'{outdir}{ct}_mC_100kb.hdf', key=context).iloc[i]
        tmp[tmp.isna()] = tmp.mean()
        tmp = np.mean([acf(tmp[bed['chrom']==c], nlags=nlag) for c in chrom_sizes.index[:22]], axis=0)
        result.append(tmp)
    result = pd.DataFrame(result, index=L1_meta.index)
    result_all.append(result)

result_all = pd.concat(result_all, axis=1)


In [ ]:
cg = sns.clustermap(result_all.drop(0, axis=1), col_cluster=False, 
                    metric='cosine', cmap='vlag', vmin=0.0, vmax=0.2, 
                    figsize=(8,8), yticklabels=result.index.map(L1_meta['L1_abbr']))
rorder = result_all.index[cg.dendrogram_row.reordered_ind]


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 6), dpi=300)
for i,context in enumerate('GATC'):
    ax = axes[i]
    tmp = result_all.loc[rorder, :].iloc[:, (i*(nlag+1)):(i+1)*(nlag+1)]
    sns.heatmap(tmp, ax=ax, cmap='vlag', cbar=False, vmin=0, vmax=0.2)
    sns.despine(ax=ax, top=False, right=False, left=False, bottom=False)
    ax.set_title(f'C{context}')
    ax.set_xticks([0, 25, 50])
    ax.set_xticklabels([f'{xx/10}M' for xx in [0, 25, 50]])
    ax.set_xlabel('Genome Distance')
    ax.set_ylabel('')
    if i==0:
        ax.set_yticks(np.arange(0.5, len(rorder)+0.5, 1))
        ax.set_yticklabels(rorder.map(L1_meta['L1_abbr']), rotation=0)
    else:
        ax.set_yticks([])

fig.tight_layout()
fig.savefig(f'{outdir}plot/autocorr_100kb_res{res}b_L1_context_heatmap.pdf', transparent=True)

In [ ]:
ncol = 5
nrow = (L1_meta.shape[0] - 1) // ncol + 1
palette = {context:c for context,c in zip('GATC', sns.color_palette('hls', len('GATC')))}
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*2, nrow*2), dpi=300, sharex='all', sharey='all')
for i,ct in enumerate(L1_meta.index):
    ax = axes.flatten()[i]
    for j,context in enumerate('GATC'):
        tmp = result_all.loc[ct, :].iloc[(j*(nlag+1)):(j+1)*(nlag+1)]
        ax.plot(np.arange(nlag+1), tmp, color=palette[context], label=f'C{context}')
    ax.set_title(L1_meta.loc[ct, 'L1_abbr'])
    ax.set_xticks([0, 25, 50])
    ax.set_xticklabels([f'{xx/10}M' for xx in [0, 25, 50]])
    ax.set_ylim([-0.02, 0.62])
    if i%ncol==0:
        ax.set_ylabel('Pearson r')
    if i>= (nrow-1)*ncol:
        ax.set_xlabel('Genome Distance')
    if i==0:
        ax.legend()

for ax in axes.flatten()[L1_meta.shape[0]:]:
    ax.axis('off')

fig.tight_layout()
fig.savefig(f'{outdir}plot/autocorr_100kb_res{res}b_context_L1_line.pdf', transparent=True)


In [ ]:
slop = 50000
ws = 500

In [ ]:
expr = pd.read_hdf(f'{indir}scRNA/pseudobulk/L1/L1.hdf', key='data')
expr.shape

In [ ]:
bed = pd.read_csv(f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz', sep='\t', header=0)
bed = bed.loc[bed['chrom'].isin(chrom_sizes.index)]
bed['gene_id_idx'] = bed['gene_id'].str.split('.').str[0]


In [ ]:
sfilter = (bed['strand']=='-')
if np.sum(sfilter)>0:
    bed['center'] = bed['start'].copy()
    bed.loc[sfilter, 'center'] = bed.loc[sfilter, 'end'].values
else:
    bed['center'] = (bed['start'] + bed['end']) // 2
    
bed = bed.loc[((bed['center'] - slop*2) > 0) & ((bed['center'] + slop*2) < bed['chrom'].map(chrom_sizes))]
chunk_size = bed.index.max() // 48
bed['chunk'] = bed.index // chunk_size
print(bed.shape[0], bed['chunk'].max())


In [ ]:
for _, chunk in bed.sample(frac=1).groupby('chunk'):
    break

In [ ]:
print(chrom, center-slop*2, center+slop*2)

In [ ]:
def matrixch(allc_path, chunk):
    with pysam.TabixFile(allc_path) as allc:
        result = {context:{} for context in 'GATC'}
        for idx,(chrom,center,strand) in chunk[['chrom', 'center', 'strand']].iterrows():
            data = {context:[] for context in 'GATCN'}
            start, end = center-slop*2, center+slop*2
            allc_lines = allc.fetch(chrom, start, end)
            for line in allc_lines:
                _, pos, _, context, mc, cov, *_ = line.split("\t")
                data[context[1]].append([pos, mc, cov])
            del data['N']
            for context in 'GATC':
                tmp = pd.DataFrame(data[context], columns=['pos', 'mc', 'cov']).astype(int)            
                posfilter = np.ones(tmp.shape[0]).astype(bool)
                for bed in rm_list:
                    bedtmp = bed[chrom][np.logical_and(bed[chrom][:,0]<end, bed[chrom][:,1]>start)]
                    for xx,yy in bedtmp:
                        posfilter[np.logical_and(tmp['pos']>=xx, tmp['pos']<=yy)] = False
                tmp = tmp.loc[posfilter]
                tmp['pos'] = tmp['pos'] - start - 1
                npos = (end - start)
                data_mc = csr_matrix((tmp['mc'], (np.zeros(tmp.shape[0]), tmp['pos'])), shape=[1, npos]).reshape((-1, ws)).sum(axis=1).A1
                data_cov = csr_matrix((tmp['cov'], (np.zeros(tmp.shape[0]), tmp['pos'])), shape=[1, npos]).reshape((-1, ws)).sum(axis=1).A1
                result[context][idx] = data_mc / data_cov
                if strand=='-':
                    result[context][idx] = result[context][idx][::-1]
        for context in 'GATC':
            result[context] = pd.DataFrame(result[context])
        return result

cpu = 24
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,chunk in bed.groupby('chunk'):
        future = executor.submit(
            matrixch,
            allc_path=allc_path, 
            chunk=chunk,
        )
        futures[future] = i

    result = {context:[] for context in 'GATC'}
    for future in as_completed(futures):
        idx = futures[future]
        tmp = future.result()
        for context in 'GATC':
            result[context].append(tmp[context])
        print(f'chunk{idx} finished')
        
for context in 'GATC':
    result[context] = pd.concat(result[context], axis=1).sort_index(axis=1)
    result[context].to_hdf(f'{outdir}{ct}_mC_geneslop100kb.hdf', key=context)
    


In [ ]:
import numpy as np

def shifted_correlations(X):
    """
    X: array of shape (200, n)
    Returns: array of shape (100,)
    """

    # ---- Step 1: fill NaNs per column ----
    col_means = np.nanmean(X, axis=0, keepdims=True)
    X = np.where(np.isnan(X), col_means, X)

    # ---- Step 2: swap dimensions to n x 200 ----
    X = X.T                                     # shape (n, 200)

    # Precompute “baseline” (rows: n, cols: 100)
    base = X[:, :100]                           # (n, 100)
    base_mean = base.mean(axis=1, keepdims=True)
    base_std  = base.std(axis=1, keepdims=True)

    out = np.zeros(100)

    for k in range(100):
        seg = X[:, k:k+100]                     # (n, 100)

        seg_mean = seg.mean(axis=1, keepdims=True)
        seg_std  = seg.std(axis=1, keepdims=True)

        # covariance across the 100 positions (axis=1)
        cov = ((seg - seg_mean) * (base - base_mean)).mean(axis=1)

        coro = cov / (seg_std * base_std)       # correlation per row
        out[k] = coro.mean()                    # average across n rows

    return out


In [ ]:
ct_name = L1_meta.loc[ct, 'L1_abbr']
idx = np.arange(100)
palette = {context:c for context,c in zip('GATC', sns.color_palette('hls', len('GATC')))}
fig, ax = plt.subplots(figsize=(3,3), dpi=300)
for i,context in enumerate('GATC'):
    ratio = result[context].T.iloc[:, 200:]
    ratio.index = bed['gene_id_idx'].values
    selr = (ratio.isna().sum(axis=1)<(0.2*ratio.shape[1]))
    ratio = ratio.loc[selr]
    selg = expr.index[expr.index.isin(ratio.index)]
    tmp = expr.loc[selg, ct_name]
    tmp = tmp[tmp>0]
    gene_group = pd.Series(0, index=selg)
    gene_group.loc[tmp.index] = pd.qcut(tmp, 4, labels=[1,2,3,4]).astype(int)
    corr = shifted_correlations(ratio.loc[gene_group.index].loc[gene_group==4].values)
    ax.plot(idx, corr, c=palette[context])


In [ ]:
nlag = 100
palette = {context:c for context,c in zip('GATC', sns.color_palette('hls', len('GATC')))}
fig, ax = plt.subplots(figsize=(3,3), dpi=300)
for i,context in enumerate('GATC'):
    tmp = result[context].loc[j]
    tmp[tmp.isna()] = tmp.mean()
    tmp = np.mean([acf(tmp[bed['chrom']==c], nlags=nlag) for c in chrom_sizes.index[:22]], axis=0)
    ax.plot(np.arange(nlag+1), tmp, color=palette[context], label=f'C{context}')

ax.set_title(num2str(res))
ax.set_xticks(np.arange(0, nlag+1, 50))
ax.set_xticklabels([f'{xx/10}M' for xx in np.arange(0, nlag+1, 50)])
if j==0:
    ax.legend()

for ax in axes[-1]:
    ax.set_xlabel('Genome Distance')

for ax in axes[:, 0]:
    ax.set_ylabel('Pearson r')

fig.tight_layout()
fig.savefig(f'{outdir}plot/{ct}_autocorr_context_res.pdf', transparent=True)
